# This notebook takes the extracted 3-4-5 sequences from programs TS_1 to TS_5 to create training datasets.

In [1]:
from sklearn.model_selection import KFold
from sklearn.model_selection import train_test_split
from matplotlib import pyplot as plt
import random
import tensorflow as tf
import numpy as np
import timeit
from sklearn import metrics
import scipy.io as sio
import csv
import os
import math 
import random as rand
from numpy import genfromtxt
from numpy import sum,isrealobj,sqrt
from numpy.random import standard_normal

#import torch
#from torch import nn

import warnings
from scipy.spatial import distance



import sys
sys.path.append("..") # Adds higher directory to python modules path.

from Modules.Tools import MapTool
from Modules.Tools import BoxPlot
from Modules.Tools import peakCorrelation as pC

In [2]:
# Signal 
params = {'legend.fontsize': 50,
          'figure.figsize': (45, 15),
         'axes.labelsize': 60,
         'axes.titlesize':70,
         'xtick.labelsize':50,
         'ytick.labelsize':50,
         'lines.linewidth': 8}
plt.rcParams.update(params)

# Notebook Parameters
- this is simply for obtaining the graphs with manual mapping. Pad is closely relate to the sampling rate for the EM signal

In [3]:
save_folder = "./Graphs/1_2_SequenceExtractor/"
load_folder = "./Datasets/Numpy/FP_GANs_LargeScale_2/full/"
pad = 8
y_low=-2.8
y_high=2.8
amount=3000

In [4]:
def getValue(dataset, loc):
    list_peaks = []
    
    for peaks in dataset:
        list_peaks.append(peaks[loc])
    return np.array(list_peaks)

# Load Extracted Sequence Datasets

In [5]:
load_Folder = "./Datasets/Extracted Sequences/"

In [6]:
TS_to_Gen_3_Seq = np.load(load_Folder + "Generating/TS_to_Gen_3_Seq.npy", allow_pickle = True)
TS_to_Gen_4_Seq = np.load(load_Folder + "Generating/TS_to_Gen_4_Seq.npy", allow_pickle = True)
TS_to_Gen_5_Seq = np.load(load_Folder + "Generating/TS_to_Gen_5_Seq.npy", allow_pickle = True)

In [7]:
Alpha_5_seq_Info = np.load(load_Folder + "Generating/Alpha_5_seq_Info.npy", allow_pickle = True)

In [8]:
TS_1_5_seq_Info = np.load(load_Folder + "Generating/TS_1_5_seq_Info.npy", allow_pickle = True)
TS_2_5_seq_Info = np.load(load_Folder + "Generating/TS_2_5_seq_Info.npy", allow_pickle = True)
TS_3_5_seq_Info = np.load(load_Folder + "Generating/TS_3_5_seq_Info.npy", allow_pickle = True)
TS_4_5_seq_Info = np.load(load_Folder + "Generating/TS_4_5_seq_Info.npy", allow_pickle = True)
TS_5_5_seq_Info = np.load(load_Folder + "Generating/TS_5_5_seq_Info.npy", allow_pickle = True)

# Creating a code embedder.

We need to be able to convert code from either string or a code document into a readable formate.
This can create several challenges.
- The amount of code can vary between samples.
- The signal size can vary.
- The embedded code sould be unique like bagging words.
- Note that words for each instruction mustnt be spaced, hypened, unscored or contain any unique lettering or symbol as this will split the word/instruction into different tokens.

Since this is just a simple experiment, we will not address all issues with this test.
Instead we will limit the signal size and amount of code to the size of desired generated program for all signals.
Additionally, we will create a precise convertion from code to numbers, where each number will represent a unique code instruction (i.e., tokenization).

In [9]:
from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer = Tokenizer(num_words=100, char_level=False)

In [10]:
Alpha_Base_Code = np.array(["ori", "sts", "lds", "ori", "mov",
        "andi", "mov", "ori", "mov", "andi"])

In [11]:
TS_1_Base_Code = np.array(["ori", "sts", "lds", "ori", "mov",
              
              "andi",
              "ori", "sts", "out", "lds", "ori",
              "sts", "lds", "ori", "mov", "andi",
              "push", "ori", "mov", "andi", "or",
              "push", "ori", "muls", "nop", "fmul",
              
              "pop",
              
              "sts", "lds", "ori", "muls", "nop",
              "sev", "sbis-f", "cp", "out", "lds",
              "mov", "com", "neg", "sbci", "tst",
              "push", "ori", "mov", "andi", "ldi",
             
              "pop",
                         
              "out", "lds", "ori", "sts",
              "cp", "brpl-f", "cli", "ori", "sts",
                         
              "lds", "ori", "sts", "bset", "push",
              "dec", "brne-t", "ldi", "add", "sub",
              "brpl-f", "cli", "ori", "sts", "lds",
                         
              "andi", "mov", "ori", "mov",
              "andi", "ori", "muls", "nop", "pop",
              
              "pop",
                         
              "ori", "sts", "ldi", "sub",
              "cln", "lds", "ori", "sts"           
              ])

TS_2_Base_Code = np.array(["lds", "ori", "sts", "out", "lds",
              
              "lds", "ori", "mov", "andi", "mov",
              "sts", "out", "lds", "ori", "sts",
              "lds", "ori", "mov", "andi", "sev",
              "ori", "mov", "andi", "or", "mov",
              "ori", "muls", "nop", "fmul", "push",
                         
              "lds", "ori", "muls", "nop", "andi",
              "sbis-f", "cp", "out", "lds", "ori",
              "com", "neg", "sbci", "tst", "andi",
              "ori", "mov", "andi", "ldi", "swap",
                         
              "swap", "add", "eor", "sub",
              "ldi", "out", "lds", "ori", "sts",
                         
              "ori", "sts", "bset", "push", "andi",
              "brne-t", "ldi", "add", "sub", "pop",
              "cli", "ori", "sts", "lds", "ori", 
              
              "pop", "lds", "ori", "sts",
              "nop", "and", "ori", "muls", "nop",
                         
              "lds", "ori", "sts", "com",
              "sen", "ori", "sts", "lds", "ori"           
              ])

TS_3_Base_Code = np.array(["ori", "sts", "lds", "ori", "mov", 
              
              "push",
                         
              "ori", "mov", "andi", "mov", "ori",
              "out", "lds", "ori", "sts", "lds",
              "ori", "mov", "andi", "sev", "sbis-f",
              "mov", "andi", "or", "mov", "com",
              "muls", "nop", "fmul", "push", "ori",
              
              "ori", "muls", "nop", "andi", "mov",
              "cp", "out", "lds", "ori", "sts",
              "neg", "sbci", "tst", "andi", "mov",
              "mov", "andi", "ldi", "swap", "add",
                         
              "andi", "mov", "ori", "mov",
              "andi", "dec", "brne-t", "ldi", "add",
                         
              "sts", "bset", "push", "andi", "mov",
              "ldi", "add", "sub", "pop", "lds",
              "ori", "sts", "lds", "ori", "pop",
                         
              "ori", "pop", "inc", "mov",
              "sbc", "jmp", "sub", "add", "nop",
                         
              "muls", "eor", "in", "tst",
              "brne-f", "fmul", "lds", "ori", "sts"           
              ])

TS_4_Base_Code = np.array(["in", "push", "ori", "mov", "andi", 
                         
              "mov", "andi", "mov", "ori", "mov",
              "lds", "ori", "sts", "lds", "ori",
              "mov", "andi", "sev", "sbis-f", "cp", 
              "andi", "or", "mov", "com", "neg",
              "nop", "fmul", "push", "ori", "mov",
               
              "muls", "nop", "andi", "mov", "ori",
              "out", "lds", "ori", "sts", "ldi",
              "sbci", "tst", "andi", "mov", "ori",
              "andi", "ldi", "swap", "add", "eor",
                         
              "lds", "ori", "sts", "bset",
              "push", "andi", "mov", "ori", "mov",
                         
              "bset", "push", "andi", "mov", "ori",
              "add", "sub", "pop", "lds", "ori",
              "sts", "lds", "ori", "pop", "inc",
                         
              "jmp", "sub", "add", "nop",
              "pop", "andi", "mov", "ori", "mov",
              
              "pop",           
              
              "andi", "mov", "ori", "mov",
              "andi", "muls", "eor", "in", "tst",           
              ])

TS_5_Base_Code = np.array(["ldi", "push", "ori", "muls", "nop", 
                         
              "andi", "mov", "ori", "mov", "andi",
              "ori", "sts", "lds", "ori", "muls",
              "andi", "sev", "sbis-f", "cp", "out",
              "or", "mov", "com", "neg", "sbci",
              "fmul", "push", "ori", "mov", "andi",
              
              "nop", "andi", "mov", "ori", "mov",
              "lds", "ori", "sts", "ldi", "sub",
              "tst", "andi", "mov", "ori", "mov",
              "ldi", "swap", "add", "eor", "sub",
               
              "andi", "mov", "ori", "mov",           
              "andi", "andi", "ori", "mov", "andi",
              
              "push", "andi", "mov", "ori", "mov",           
              "sub", "pop", "lds", "ori", "sts",           
              "lds", "ori", "pop", "inc", "mov",
              
              "pop",
                         
              "and", "ori", "muls", "nop",
              "out", "lds", "ori", "sts", "com"           
              ])

In [12]:
'''
Simply adds the number of nops to the array
'''
def addNops(number, array):
    code_list = []
    count = 0
    while count < number:
        code_list.append('nop')
        count = count+1
    
    for code in array:
        code_list.append(code)
    return np.array(code_list)

In [13]:
Alpha_Code_5_Seq = addNops(4, Alpha_Base_Code)
Bravo_Code_5_Seq = addNops(4, Bravo_Base_Code)
Charlie_Code_5_Seq = addNops(4, Charlie_Base_Code)
Delta_Code_5_Seq = addNops(4, Delta_Base_Code)
Echo_Code_5_Seq = addNops(4, Echo_Base_Code)

In [14]:
TS_1_5_Seq = addNops(4, TS_1_Base_Code)
TS_2_5_Seq = addNops(4, TS_2_Base_Code)
TS_3_5_Seq = addNops(4, TS_3_Base_Code)
TS_4_5_Seq = addNops(4, TS_4_Base_Code)
TS_5_5_Seq = addNops(4, TS_5_Base_Code)

In [15]:
'''
Simply splits known multi cycle codes.
'''
def splitCode(array):
    code_list = []
    two_cycle_list = ["sts", "lds", "muls", "fmul", "push", "pop"]
    three_cycle_list = ["jmp"]
    for code in array:
        # check for hyphens
        if code == "brne-f":
            code_list.append("brneF")
        elif code == "brne-t":
            code_list.append("brneTPT1")
            code_list.append("brneTPT2")
        elif code == "brpl-f":
            code_list.append("brplF")
        elif code == "sbis-f":
            code_list.append("sbisF")
        
        # check for two cycles
        elif code in two_cycle_list:
            code_list.append(code + "PT1")
            code_list.append(code + "PT2")
            
        # check for thee cycles
        elif code in three_cycle_list:
            code_list.append(code + "PT1")
            code_list.append(code + "PT2")
            code_list.append(code + "PT3")
        
        else:
            code_list.append(code)
            
    return np.array(code_list)

In [16]:
Alpha_Code_5_Seq_Split = splitCode(Alpha_Code_5_Seq)
Bravo_Code_5_Seq_Split = splitCode(Bravo_Code_5_Seq)
Charlie_Code_5_Seq_Split = splitCode(Charlie_Code_5_Seq)
Delta_Code_5_Seq_Split = splitCode(Delta_Code_5_Seq)
Echo_Code_5_Seq_Split = splitCode(Echo_Code_5_Seq)

In [17]:
TS_1_5_Seq_Split = splitCode(TS_1_5_Seq)
TS_2_5_Seq_Split = splitCode(TS_2_5_Seq)
TS_3_5_Seq_Split = splitCode(TS_3_5_Seq)
TS_4_5_Seq_Split = splitCode(TS_4_5_Seq)
TS_5_5_Seq_Split = splitCode(TS_5_5_Seq)

In [18]:
all_code = np.copy(Alpha_Code_5_Seq_Split)
all_code = np.concatenate((all_code, Bravo_Code_5_Seq_Split))
all_code = np.concatenate((all_code, Charlie_Code_5_Seq_Split))
all_code = np.concatenate((all_code, Delta_Code_5_Seq_Split))
all_code = np.concatenate((all_code, Echo_Code_5_Seq_Split))

all_code = np.concatenate((all_code, TS_1_5_Seq_Split))
all_code = np.concatenate((all_code, TS_2_5_Seq_Split))
all_code = np.concatenate((all_code, TS_3_5_Seq_Split))
all_code = np.concatenate((all_code, TS_4_5_Seq_Split))
all_code = np.concatenate((all_code, TS_5_5_Seq_Split))

In [19]:
print(Alpha_Code_5_Seq_Split.shape)
print(Bravo_Code_5_Seq_Split.shape)
print(Charlie_Code_5_Seq_Split.shape)
print(Delta_Code_5_Seq_Split.shape)
print(Echo_Code_5_Seq_Split.shape)

(16,)
(30,)
(43,)
(54,)
(70,)


In [20]:
print(all_code)
print(all_code.shape)

['nop' 'nop' 'nop' 'nop' 'ori' 'stsPT1' 'stsPT2' 'ldsPT1' 'ldsPT2' 'ori'
 'mov' 'andi' 'mov' 'ori' 'mov' 'andi' 'nop' 'nop' 'nop' 'nop' 'ldsPT1'
 'ldsPT2' 'ori' 'stsPT1' 'stsPT2' 'out' 'ldsPT1' 'ldsPT2' 'ori' 'stsPT1'
 'stsPT2' 'ldsPT1' 'ldsPT2' 'ori' 'mulsPT1' 'mulsPT2' 'nop' 'andi' 'mov'
 'ori' 'mov' 'andi' 'andi' 'ori' 'mov' 'andi' 'nop' 'nop' 'nop' 'nop'
 'ori' 'stsPT1' 'stsPT2' 'ldsPT1' 'ldsPT2' 'ori' 'mov' 'andi' 'sev'
 'sbisF' 'cp' 'out' 'ldsPT1' 'ldsPT2' 'ori' 'stsPT1' 'stsPT2' 'ldi' 'sub'
 'cln' 'ldsPT1' 'ldsPT2' 'ori' 'stsPT1' 'stsPT2' 'bset' 'pushPT1'
 'pushPT2' 'andi' 'mov' 'ori' 'mov' 'andi' 'ori' 'mulsPT1' 'mulsPT2' 'nop'
 'popPT1' 'popPT2' 'nop' 'nop' 'nop' 'nop' 'in' 'pushPT1' 'pushPT2' 'ori'
 'mov' 'andi' 'or' 'mov' 'com' 'neg' 'sbci' 'tst' 'andi' 'mov' 'ori' 'mov'
 'andi' 'dec' 'brneTPT1' 'brneTPT2' 'ldi' 'add' 'sub' 'popPT1' 'popPT2'
 'ldsPT1' 'ldsPT2' 'ori' 'stsPT1' 'stsPT2' 'nop' 'and' 'ori' 'mulsPT1'
 'mulsPT2' 'nop' 'out' 'ldsPT1' 'ldsPT2' 'ori' 'stsPT1' 'stsPT2'

In [21]:
all_code_unique = np.unique(all_code)
print(all_code_unique)

['add' 'and' 'andi' 'brneF' 'brneTPT1' 'brneTPT2' 'brplF' 'bset' 'cli'
 'cln' 'com' 'cp' 'dec' 'eor' 'fmulPT1' 'fmulPT2' 'in' 'inc' 'jmpPT1'
 'jmpPT2' 'jmpPT3' 'ldi' 'ldsPT1' 'ldsPT2' 'mov' 'mulsPT1' 'mulsPT2' 'neg'
 'nop' 'or' 'ori' 'out' 'popPT1' 'popPT2' 'pushPT1' 'pushPT2' 'sbc' 'sbci'
 'sbisF' 'sen' 'sev' 'stsPT1' 'stsPT2' 'sub' 'swap' 'tst']


In [22]:
string_all_code_unique = ""
First = True
for inst in all_code_unique:
    if First:
        string_all_code_unique = string_all_code_unique + inst
        First = False
    else:
        string_all_code_unique = string_all_code_unique + ", " + inst
print(string_all_code_unique)

add, and, andi, brneF, brneTPT1, brneTPT2, brplF, bset, cli, cln, com, cp, dec, eor, fmulPT1, fmulPT2, in, inc, jmpPT1, jmpPT2, jmpPT3, ldi, ldsPT1, ldsPT2, mov, mulsPT1, mulsPT2, neg, nop, or, ori, out, popPT1, popPT2, pushPT1, pushPT2, sbc, sbci, sbisF, sen, sev, stsPT1, stsPT2, sub, swap, tst


In [23]:
temp = np.array([string_all_code_unique])

In [24]:
print(temp)

['add, and, andi, brneF, brneTPT1, brneTPT2, brplF, bset, cli, cln, com, cp, dec, eor, fmulPT1, fmulPT2, in, inc, jmpPT1, jmpPT2, jmpPT3, ldi, ldsPT1, ldsPT2, mov, mulsPT1, mulsPT2, neg, nop, or, ori, out, popPT1, popPT2, pushPT1, pushPT2, sbc, sbci, sbisF, sen, sev, stsPT1, stsPT2, sub, swap, tst']


# save this to load for later to get same tokenizer settings.

In [30]:
np.save(".\Datasets\FP_GANs_LargeScale_2\k-Fold\TrainingDatasets\All_Code.npy", temp)

In [25]:
all_instructions = string_all_code_unique

sentence_Code = [all_instructions]
tokenizer.fit_on_texts(sentence_Code)

In [26]:
print(tokenizer.texts_to_sequences(sentence_Code))

[[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46]]


In [27]:
print(tokenizer.texts_to_sequences(['jmpPT1']))

[[19]]


## Create benign case code

Note that we have full knowledge of the code of the real programs. As such we can full the information of the code taking into acount the max number of sequence the model accounts for.
- Overall this will end up being the same code sequence as the 5 sequence version for the program we are attempting to generate

In [28]:
program_Code_Alpha = Alpha_Code_5_Seq_Split
print(program_Code_Alpha)

['nop' 'nop' 'nop' 'nop' 'ori' 'stsPT1' 'stsPT2' 'ldsPT1' 'ldsPT2' 'ori'
 'mov' 'andi' 'mov' 'ori' 'mov' 'andi']


In [29]:
def createBaseCode(Info):
    Base = []
    for seq in Info:
        code = seq["Code"]
        tokens = tokenizer.texts_to_sequences(code)
        token_list = []
        for token in tokens:
            token_list.append(token[0])
        Base.append(token_list)
    return np.array(Base)

In [30]:
for seq in Alpha_5_seq_Info:
    print(seq["Code"])

['nop' 'nop' 'nop' 'nop' 'ori']
['nop' 'nop' 'nop' 'ori' 'stsPT1']
['nop' 'nop' 'ori' 'stsPT1' 'stsPT2']
['nop' 'ori' 'stsPT1' 'stsPT2' 'ldsPT1']
['ori' 'stsPT1' 'stsPT2' 'ldsPT1' 'ldsPT2']
['stsPT1' 'stsPT2' 'ldsPT1' 'ldsPT2' 'ori']
['stsPT2' 'ldsPT1' 'ldsPT2' 'ori' 'mov']
['ldsPT1' 'ldsPT2' 'ori' 'mov' 'andi']
['ldsPT2' 'ori' 'mov' 'andi' 'mov']
['ori' 'mov' 'andi' 'mov' 'ori']
['mov' 'andi' 'mov' 'ori' 'mov']
['andi' 'mov' 'ori' 'mov' 'andi']


In [31]:
Alpha_Base_Code = createBaseCode(Alpha_5_seq_Info)

In [32]:
print(Alpha_Base_Code)

[[29 29 29 29 31]
 [29 29 29 31 42]
 [29 29 31 42 43]
 [29 31 42 43 23]
 [31 42 43 23 24]
 [42 43 23 24 31]
 [43 23 24 31 25]
 [23 24 31 25  3]
 [24 31 25  3 25]
 [31 25  3 25 31]
 [25  3 25 31 25]
 [ 3 25 31 25  3]]


In [42]:
np.save(".\Datasets\FP_GANs_LargeScale_2\k-Fold\TrainingDatasets\Alpha_Base_code.npy", Alpha_Base_Code)

## Training set base code

In [29]:
TS_1_Base_Code = createBaseCode(TS_1_5_seq_Info)
print(TS_1_Base_Code)
np.save(".\Datasets\FP_GANs_LargeScale_2\k-Fold\TrainingDatasets\TS_1_Base_code.npy", TS_1_Base_Code)

[[29 29 29 29 31]
 [29 29 29 31 42]
 [29 29 31 42 43]
 [29 31 42 43 23]
 [31 42 43 23 24]
 [42 43 23 24 31]
 [43 23 24 31 25]
 [23 24 31 25  3]
 [24 31 25  3 31]
 [31 25  3 31 42]
 [25  3 31 42 43]
 [ 3 31 42 43 32]
 [31 42 43 32 23]
 [42 43 32 23 24]
 [43 32 23 24 31]
 [32 23 24 31 42]
 [23 24 31 42 43]
 [24 31 42 43 23]
 [31 42 43 23 24]
 [42 43 23 24 31]
 [43 23 24 31 25]
 [23 24 31 25  3]
 [24 31 25  3 35]
 [31 25  3 35 36]
 [25  3 35 36 31]
 [ 3 35 36 31 25]
 [35 36 31 25  3]
 [36 31 25  3 30]
 [31 25  3 30 35]
 [25  3 30 35 36]
 [ 3 30 35 36 31]
 [30 35 36 31 26]
 [35 36 31 26 27]
 [36 31 26 27 29]
 [31 26 27 29 15]
 [26 27 29 15 16]
 [27 29 15 16 33]
 [29 15 16 33 34]
 [15 16 33 34 42]
 [16 33 34 42 43]
 [33 34 42 43 23]
 [34 42 43 23 24]
 [42 43 23 24 31]
 [43 23 24 31 26]
 [23 24 31 26 27]
 [24 31 26 27 29]
 [31 26 27 29 41]
 [26 27 29 41 39]
 [27 29 41 39 12]
 [29 41 39 12 32]
 [41 39 12 32 23]
 [39 12 32 23 24]
 [12 32 23 24 25]
 [32 23 24 25 11]
 [23 24 25 11 28]
 [24 25 11

In [30]:
TS_2_Base_Code = createBaseCode(TS_2_5_seq_Info)
print(TS_2_Base_Code)
np.save(".\Datasets\FP_GANs_LargeScale_2\k-Fold\TrainingDatasets\TS_2_Base_code.npy", TS_2_Base_Code)

[[29 29 29 29 23]
 [29 29 29 23 24]
 [29 29 23 24 31]
 [29 23 24 31 42]
 [23 24 31 42 43]
 [24 31 42 43 32]
 [31 42 43 32 23]
 [42 43 32 23 24]
 [43 32 23 24 23]
 [32 23 24 23 24]
 [23 24 23 24 31]
 [24 23 24 31 25]
 [23 24 31 25  3]
 [24 31 25  3 25]
 [31 25  3 25 42]
 [25  3 25 42 43]
 [ 3 25 42 43 32]
 [25 42 43 32 23]
 [42 43 32 23 24]
 [43 32 23 24 31]
 [32 23 24 31 42]
 [23 24 31 42 43]
 [24 31 42 43 23]
 [31 42 43 23 24]
 [42 43 23 24 31]
 [43 23 24 31 25]
 [23 24 31 25  3]
 [24 31 25  3 41]
 [31 25  3 41 31]
 [25  3 41 31 25]
 [ 3 41 31 25  3]
 [41 31 25  3 30]
 [31 25  3 30 25]
 [25  3 30 25 31]
 [ 3 30 25 31 26]
 [30 25 31 26 27]
 [25 31 26 27 29]
 [31 26 27 29 15]
 [26 27 29 15 16]
 [27 29 15 16 35]
 [29 15 16 35 36]
 [15 16 35 36 23]
 [16 35 36 23 24]
 [35 36 23 24 31]
 [36 23 24 31 26]
 [23 24 31 26 27]
 [24 31 26 27 29]
 [31 26 27 29  3]
 [26 27 29  3 39]
 [27 29  3 39 12]
 [29  3 39 12 32]
 [ 3 39 12 32 23]
 [39 12 32 23 24]
 [12 32 23 24 31]
 [32 23 24 31 11]
 [23 24 31

In [31]:
TS_3_Base_Code = createBaseCode(TS_3_5_seq_Info)
print(TS_3_Base_Code)
np.save(".\Datasets\FP_GANs_LargeScale_2\k-Fold\TrainingDatasets\TS_3_Base_code.npy", TS_3_Base_Code)

[[29 29 29 29 31]
 [29 29 29 31 42]
 [29 29 31 42 43]
 [29 31 42 43 23]
 [31 42 43 23 24]
 [42 43 23 24 31]
 [43 23 24 31 25]
 [23 24 31 25 35]
 [24 31 25 35 36]
 [31 25 35 36 31]
 [25 35 36 31 25]
 [35 36 31 25  3]
 [36 31 25  3 25]
 [31 25  3 25 31]
 [25  3 25 31 32]
 [ 3 25 31 32 23]
 [25 31 32 23 24]
 [31 32 23 24 31]
 [32 23 24 31 42]
 [23 24 31 42 43]
 [24 31 42 43 23]
 [31 42 43 23 24]
 [42 43 23 24 31]
 [43 23 24 31 25]
 [23 24 31 25  3]
 [24 31 25  3 41]
 [31 25  3 41 39]
 [25  3 41 39 25]
 [ 3 41 39 25  3]
 [41 39 25  3 30]
 [39 25  3 30 25]
 [25  3 30 25 11]
 [ 3 30 25 11 26]
 [30 25 11 26 27]
 [25 11 26 27 29]
 [11 26 27 29 15]
 [26 27 29 15 16]
 [27 29 15 16 35]
 [29 15 16 35 36]
 [15 16 35 36 31]
 [16 35 36 31 31]
 [35 36 31 31 26]
 [36 31 31 26 27]
 [31 31 26 27 29]
 [31 26 27 29  3]
 [26 27 29  3 25]
 [27 29  3 25 12]
 [29  3 25 12 32]
 [ 3 25 12 32 23]
 [25 12 32 23 24]
 [12 32 23 24 31]
 [32 23 24 31 42]
 [23 24 31 42 43]
 [24 31 42 43 28]
 [31 42 43 28 38]
 [42 43 28

In [32]:
TS_4_Base_Code = createBaseCode(TS_4_5_seq_Info)
print(TS_4_Base_Code)
np.save(".\Datasets\FP_GANs_LargeScale_2\k-Fold\TrainingDatasets\TS_4_Base_code.npy", TS_4_Base_Code)

[[29 29 29 29 17]
 [29 29 29 17 35]
 [29 29 17 35 36]
 [29 17 35 36 31]
 [17 35 36 31 25]
 [35 36 31 25  3]
 [36 31 25  3 25]
 [31 25  3 25  3]
 [25  3 25  3 25]
 [ 3 25  3 25 31]
 [25  3 25 31 25]
 [ 3 25 31 25 23]
 [25 31 25 23 24]
 [31 25 23 24 31]
 [25 23 24 31 42]
 [23 24 31 42 43]
 [24 31 42 43 23]
 [31 42 43 23 24]
 [42 43 23 24 31]
 [43 23 24 31 25]
 [23 24 31 25  3]
 [24 31 25  3 41]
 [31 25  3 41 39]
 [25  3 41 39 12]
 [ 3 41 39 12  3]
 [41 39 12  3 30]
 [39 12  3 30 25]
 [12  3 30 25 11]
 [ 3 30 25 11 28]
 [30 25 11 28 29]
 [25 11 28 29 15]
 [11 28 29 15 16]
 [28 29 15 16 35]
 [29 15 16 35 36]
 [15 16 35 36 31]
 [16 35 36 31 25]
 [35 36 31 25 26]
 [36 31 25 26 27]
 [31 25 26 27 29]
 [25 26 27 29  3]
 [26 27 29  3 25]
 [27 29  3 25 31]
 [29  3 25 31 32]
 [ 3 25 31 32 23]
 [25 31 32 23 24]
 [31 32 23 24 31]
 [32 23 24 31 42]
 [23 24 31 42 43]
 [24 31 42 43 22]
 [31 42 43 22 38]
 [42 43 22 38 46]
 [43 22 38 46  3]
 [22 38 46  3 25]
 [38 46  3 25 31]
 [46  3 25 31  3]
 [ 3 25 31

In [33]:
TS_5_Base_Code = createBaseCode(TS_5_5_seq_Info)
print(TS_5_Base_Code)
np.save(".\Datasets\FP_GANs_LargeScale_2\k-Fold\TrainingDatasets\TS_5_Base_code.npy", TS_5_Base_Code)

[[29 29 29 29 22]
 [29 29 29 22 35]
 [29 29 22 35 36]
 [29 22 35 36 31]
 [22 35 36 31 26]
 [35 36 31 26 27]
 [36 31 26 27 29]
 [31 26 27 29  3]
 [26 27 29  3 25]
 [27 29  3 25 31]
 [29  3 25 31 25]
 [ 3 25 31 25  3]
 [25 31 25  3 31]
 [31 25  3 31 42]
 [25  3 31 42 43]
 [ 3 31 42 43 23]
 [31 42 43 23 24]
 [42 43 23 24 31]
 [43 23 24 31 26]
 [23 24 31 26 27]
 [24 31 26 27  3]
 [31 26 27  3 41]
 [26 27  3 41 39]
 [27  3 41 39 12]
 [ 3 41 39 12 32]
 [41 39 12 32 30]
 [39 12 32 30 25]
 [12 32 30 25 11]
 [32 30 25 11 28]
 [30 25 11 28 38]
 [25 11 28 38 15]
 [11 28 38 15 16]
 [28 38 15 16 35]
 [38 15 16 35 36]
 [15 16 35 36 31]
 [16 35 36 31 25]
 [35 36 31 25  3]
 [36 31 25  3 29]
 [31 25  3 29  3]
 [25  3 29  3 25]
 [ 3 29  3 25 31]
 [29  3 25 31 25]
 [ 3 25 31 25 23]
 [25 31 25 23 24]
 [31 25 23 24 31]
 [25 23 24 31 42]
 [23 24 31 42 43]
 [24 31 42 43 22]
 [31 42 43 22 44]
 [42 43 22 44 46]
 [43 22 44 46  3]
 [22 44 46  3 25]
 [44 46  3 25 31]
 [46  3 25 31 25]
 [ 3 25 31 25 22]
 [25 31 25

# Creating TrainingSet

In [54]:
save_Dataset_Folder = "./Datasets/FP_GANs_LargeScale_2/k-fold/TrainingDatasets/"

## TS - Full

In [55]:
sequence = 0
for info in TS_to_Gen_3_Seq:
    print("Sequence: ", sequence)
    print(info["Code"])
    print(info["Seq"].shape)
    print("")
    sequence = sequence +1
    
print("==============================================")
print("Total differing sequences: ", len(TS_to_Gen_3_Seq))

Sequence:  0
['nop' 'nop' 'ori']
(6000, 1)

Sequence:  1
['nop' 'ori' 'stsPT1']
(6000, 1)

Sequence:  2
['ori' 'stsPT1' 'stsPT2']
(90000, 1)

Sequence:  3
['stsPT1' 'stsPT2' 'ldsPT1']
(45000, 1)

Sequence:  4
['stsPT2' 'ldsPT1' 'ldsPT2']
(45000, 1)

Sequence:  5
['ldsPT1' 'ldsPT2' 'ori']
(108000, 1)

Sequence:  6
['ldsPT2' 'ori' 'mov']
(21000, 1)

Sequence:  7
['ori' 'mov' 'andi']
(57000, 1)

Sequence:  8
['mov' 'andi' 'ori']
(9000, 1)

Sequence:  9
['andi' 'ori' 'stsPT1']
(6000, 1)

Sequence:  10
['stsPT1' 'stsPT2' 'out']
(9000, 1)

Sequence:  11
['stsPT2' 'out' 'ldsPT1']
(9000, 1)

Sequence:  12
['out' 'ldsPT1' 'ldsPT2']
(33000, 1)

Sequence:  13
['ldsPT2' 'ori' 'stsPT1']
(60000, 1)

Sequence:  14
['mov' 'andi' 'pushPT1']
(6000, 1)

Sequence:  15
['andi' 'pushPT1' 'pushPT2']
(6000, 1)

Sequence:  16
['pushPT1' 'pushPT2' 'ori']
(27000, 1)

Sequence:  17
['pushPT2' 'ori' 'mov']
(18000, 1)

Sequence:  18
['mov' 'andi' 'or']
(9000, 1)

Sequence:  19
['andi' 'or' 'pushPT1']
(3000, 1)

Seq

In [56]:
sequence = 0
for info in TS_to_Gen_4_Seq:
    print("Sequence: ", sequence)
    print(info["Code"])
    print(info["Seq"].shape)
    print("")
    sequence = sequence +1
    
print("==============================================")
print("Total differing sequences: ", len(TS_to_Gen_4_Seq))

Sequence:  0
['nop' 'nop' 'nop' 'ori']
(6000, 1)

Sequence:  1
['nop' 'nop' 'ori' 'stsPT1']
(6000, 1)

Sequence:  2
['nop' 'ori' 'stsPT1' 'stsPT2']
(6000, 1)

Sequence:  3
['ori' 'stsPT1' 'stsPT2' 'ldsPT1']
(42000, 1)

Sequence:  4
['stsPT1' 'stsPT2' 'ldsPT1' 'ldsPT2']
(45000, 1)

Sequence:  5
['stsPT2' 'ldsPT1' 'ldsPT2' 'ori']
(42000, 1)

Sequence:  6
['ldsPT1' 'ldsPT2' 'ori' 'mov']
(21000, 1)

Sequence:  7
['ldsPT2' 'ori' 'mov' 'andi']
(18000, 1)

Sequence:  8
['ori' 'mov' 'andi' 'ori']
(9000, 1)

Sequence:  9
['mov' 'andi' 'ori' 'stsPT1']
(6000, 1)

Sequence:  10
['andi' 'ori' 'stsPT1' 'stsPT2']
(6000, 1)

Sequence:  11
['ori' 'stsPT1' 'stsPT2' 'out']
(6000, 1)

Sequence:  12
['stsPT1' 'stsPT2' 'out' 'ldsPT1']
(9000, 1)

Sequence:  13
['stsPT2' 'out' 'ldsPT1' 'ldsPT2']
(9000, 1)

Sequence:  14
['out' 'ldsPT1' 'ldsPT2' 'ori']
(27000, 1)

Sequence:  15
['ldsPT1' 'ldsPT2' 'ori' 'stsPT1']
(60000, 1)

Sequence:  16
['ldsPT2' 'ori' 'stsPT1' 'stsPT2']
(60000, 1)

Sequence:  17
['ori' 'mov'

In [57]:
sequence = 0
for info in TS_to_Gen_5_Seq:
    print("Sequence: ", sequence)
    print(info["Code"])
    print(info["Seq"].shape)
    print("")
    sequence = sequence +1
    
print("==============================================")
print("Total differing sequences: ", len(TS_to_Gen_5_Seq))

Sequence:  0
['nop' 'nop' 'nop' 'nop' 'ori']
(6000, 1)

Sequence:  1
['nop' 'nop' 'nop' 'ori' 'stsPT1']
(6000, 1)

Sequence:  2
['nop' 'nop' 'ori' 'stsPT1' 'stsPT2']
(6000, 1)

Sequence:  3
['nop' 'ori' 'stsPT1' 'stsPT2' 'ldsPT1']
(6000, 1)

Sequence:  4
['ori' 'stsPT1' 'stsPT2' 'ldsPT1' 'ldsPT2']
(42000, 1)

Sequence:  5
['stsPT1' 'stsPT2' 'ldsPT1' 'ldsPT2' 'ori']
(42000, 1)

Sequence:  6
['stsPT2' 'ldsPT1' 'ldsPT2' 'ori' 'mov']
(18000, 1)

Sequence:  7
['ldsPT1' 'ldsPT2' 'ori' 'mov' 'andi']
(18000, 1)

Sequence:  8
['ldsPT2' 'ori' 'mov' 'andi' 'ori']
(3000, 1)

Sequence:  9
['ori' 'mov' 'andi' 'ori' 'stsPT1']
(6000, 1)

Sequence:  10
['mov' 'andi' 'ori' 'stsPT1' 'stsPT2']
(6000, 1)

Sequence:  11
['andi' 'ori' 'stsPT1' 'stsPT2' 'out']
(3000, 1)

Sequence:  12
['ori' 'stsPT1' 'stsPT2' 'out' 'ldsPT1']
(6000, 1)

Sequence:  13
['stsPT1' 'stsPT2' 'out' 'ldsPT1' 'ldsPT2']
(9000, 1)

Sequence:  14
['stsPT2' 'out' 'ldsPT1' 'ldsPT2' 'ori']
(6000, 1)

Sequence:  15
['out' 'ldsPT1' 'ldsPT2' 'o

- Note: we need a training set that combines all three sequences lengths.

In [58]:
'''
Copy over as a list object.
Need a list to add to other dictionary items to...
'''
def copy2list(Dataset):
    new_list = []
    for info in Dataset:
        new_list.append(info)
    return new_list

In [59]:
'''
Combines the dictionary dataset to the first Dataset
'''
def appendTrainingDataset(new_list, Dataset):
    for info in Dataset:
        new_list.append(info)
    return new_list

# Create the combined sets for each program

In [60]:
TS_to_Gen = copy2list(TS_to_Gen_3_Seq)
TS_to_Gen = appendTrainingDataset(TS_to_Gen, TS_to_Gen_4_Seq)
TS_to_Gen = appendTrainingDataset(TS_to_Gen, TS_to_Gen_5_Seq)
TS_to_Gen = np.array(TS_to_Gen)

np.save(save_Dataset_Folder + "TS_to_Gen.npy", TS_to_Gen)

In [61]:
sequence = 0
for info in TS_to_Gen:
    print("Sequence: ", sequence)
    print(info["Code"])
    print(info["Seq"].shape)
    print("")
    sequence = sequence + 1
    
print("==============================================")
print("Total differing sequences: ", len(TS_to_Gen))

Sequence:  0
['nop' 'nop' 'ori']
(6000, 1)

Sequence:  1
['nop' 'ori' 'stsPT1']
(6000, 1)

Sequence:  2
['ori' 'stsPT1' 'stsPT2']
(90000, 1)

Sequence:  3
['stsPT1' 'stsPT2' 'ldsPT1']
(45000, 1)

Sequence:  4
['stsPT2' 'ldsPT1' 'ldsPT2']
(45000, 1)

Sequence:  5
['ldsPT1' 'ldsPT2' 'ori']
(108000, 1)

Sequence:  6
['ldsPT2' 'ori' 'mov']
(21000, 1)

Sequence:  7
['ori' 'mov' 'andi']
(57000, 1)

Sequence:  8
['mov' 'andi' 'ori']
(9000, 1)

Sequence:  9
['andi' 'ori' 'stsPT1']
(6000, 1)

Sequence:  10
['stsPT1' 'stsPT2' 'out']
(9000, 1)

Sequence:  11
['stsPT2' 'out' 'ldsPT1']
(9000, 1)

Sequence:  12
['out' 'ldsPT1' 'ldsPT2']
(33000, 1)

Sequence:  13
['ldsPT2' 'ori' 'stsPT1']
(60000, 1)

Sequence:  14
['mov' 'andi' 'pushPT1']
(6000, 1)

Sequence:  15
['andi' 'pushPT1' 'pushPT2']
(6000, 1)

Sequence:  16
['pushPT1' 'pushPT2' 'ori']
(27000, 1)

Sequence:  17
['pushPT2' 'ori' 'mov']
(18000, 1)

Sequence:  18
['mov' 'andi' 'or']
(9000, 1)

Sequence:  19
['andi' 'or' 'pushPT1']
(3000, 1)

Seq

## Now we just need to convert to tokens and split to inputs and targets.

In [62]:
'''
This will take the code and sequence information and convert it into training set to run GANs on.
'''
def convertToGANsTrainSet(Info, tokenizer, max_seq):
    first = True
    for info in Info:
        seqs = info["Seq"]
        code = info["Code"].tolist()
        code_tok = tokenizer.texts_to_sequences([code])
        print(code_tok)
        # full with null value to indicate cut off if code sequence is less thatn max_seq
        is_correct_length = False
        while is_correct_length == False:
            if len(code_tok[0]) == max_seq:
                is_correct_length = True
            else:
                null_code = [-1]
                code_tok[0] = null_code + code_tok[0]
        code_tok = np.array(code_tok)
        
        codes = np.tile(code_tok, (seqs.shape[0],1))
        
        if first:
            all_seq = np.array(seqs)
            all_code = np.array(codes)
            first = False
        
        else:
            all_seq = np.concatenate((all_seq, seqs))
            all_code = np.concatenate((all_code, codes))
    
    all_seq = np.array(all_seq)
    all_code = np.array(all_code)
    
    return all_seq, all_code

In [63]:
'''
This will take the code and sequence information and convert it into training set to run GANs on.
Keeps the samples from each instance the same number.
'''
def convertToGANsTrainSet_Balance(Info, tokenizer, max_seq, amount, random_seed=20):
    first = True
    for info in Info:
        seqs = np.copy(info["Seq"])
        np.random.seed(random_seed)
        random.shuffle(seqs)
        seqs = seqs[:amount]
        
        code = info["Code"].tolist()
        code_tok = tokenizer.texts_to_sequences([code])
        # full with null value to indicate cut off if code sequence is less thatn max_seq
        is_correct_length = False
        while is_correct_length == False:
            if len(code_tok[0]) == max_seq:
                is_correct_length = True
            else:
                null_code = [-1]
                code_tok[0] = null_code + code_tok[0]
        code_tok = np.array(code_tok)
        
        codes = np.tile(code_tok, (seqs.shape[0],1))
        
        if first:
            all_seq = np.array(seqs)
            all_code = np.array(codes)
            first = False
        
        else:
            all_seq = np.concatenate((all_seq, seqs))
            all_code = np.concatenate((all_code, codes))
    
    all_seq = np.array(all_seq)
    all_code = np.array(all_code)
    
    return all_seq, all_code

In [64]:
all_seq, all_code = convertToGANsTrainSet_Balance(TS_to_Gen, tokenizer, 5, 3000)

In [68]:
area = 425
section = 3000*area
amount = 10

print(all_seq[section:section+amount])
print(all_code[section:section+amount])

[[1.89603235]
 [1.89603235]
 [1.89603235]
 [1.89603235]
 [1.89603235]
 [1.89603235]
 [1.89603235]
 [1.89603235]
 [1.89603235]
 [1.86594345]]
[[-1 16 35 36 31]
 [-1 16 35 36 31]
 [-1 16 35 36 31]
 [-1 16 35 36 31]
 [-1 16 35 36 31]
 [-1 16 35 36 31]
 [-1 16 35 36 31]
 [-1 16 35 36 31]
 [-1 16 35 36 31]
 [-1 16 35 36 31]]


In [69]:
3000 * 986

2958000

In [67]:
all_seq.shape

(2958000, 1)

# Create trainingset for differeing percent.
- We need to create training set that will have 90% and 80% seen data for programs Alpha, Bravo, Charlie, Delta, and Echo.
- As such, we will create a method to remove given sequences from the created full Trainingset above.

In [33]:
Alpha_Base_Code = np.array(["ori", "sts", "lds", "ori", "mov",
        "andi", "mov", "ori", "mov", "andi"])

In [34]:
'''
Full_TS: The Full training set to remove sequences from.
Remove_Sets: Sets of sequences to remove. Accepts 5 sequences values and removes the 5,4 and 3 sequences.
'''
def CreateNewTS(Full_TS, Remove_Sets):
    All_Remove_Sets = createAllRemove_Set(Remove_Sets)
    #print(All_Remove_Sets)
    
    new_Info = []
    print("Removed Sequences")
    for Info in Full_TS:
        if checkSequences(Info["Code"], All_Remove_Sets):
            # add the instances of the obtained info to Info
            dictSeq = {"Seq": Info["Seq"], "Code": Info["Code"]}
            new_Info.append(dictSeq)
            
    return new_Info

In [35]:
def createAllRemove_Set(Remove_Set):
    All_Remove_Set = []
    for Sequence in Remove_Set:
        three = []
        four = []
        five = []
        for i in range(len(Sequence)):
            if i == 0:
                five.append(Sequence[i])
            if i == 1:
                four.append(Sequence[i])
            if i == 2:
                three.append(Sequence[i])
            if i > 0:
                five.append(Sequence[i])
            if i > 1:
                four.append(Sequence[i])
            if i > 2:
                three.append(Sequence[i])
        All_Remove_Set.append(np.array(three))
        All_Remove_Set.append(np.array(four))
        All_Remove_Set.append(np.array(five))
    
    return All_Remove_Set

In [36]:
def checkSequences(sequence, Remove_Set):
    for sequence_remove in Remove_Set:
        #print(sequence_remove)
        #print(sequence)
        #print(sequence.shape)
        if sequence.shape[0] == sequence_remove.shape[0]:
            if (sequence == sequence_remove).all():
                print(sequence)
                return False
    
    return True

In [37]:
string_all_code_unique = ""
First = True
for inst in all_code_unique:
    if First:
        string_all_code_unique = string_all_code_unique + inst
        First = False
    else:
        string_all_code_unique = string_all_code_unique + ", " + inst
print(string_all_code_unique)

add, and, andi, brneF, brneTPT1, brneTPT2, brplF, bset, cli, cln, com, cp, dec, eor, fmulPT1, fmulPT2, in, inc, jmpPT1, jmpPT2, jmpPT3, ldi, ldsPT1, ldsPT2, mov, mulsPT1, mulsPT2, neg, nop, or, ori, out, popPT1, popPT2, pushPT1, pushPT2, sbc, sbci, sbisF, sen, sev, stsPT1, stsPT2, sub, swap, tst


## Alpha

In [120]:
Remove_Sets = [["ldsPT1", "ldsPT2", "ori", "mov", "andi"]]

TS_to_Gen_alpha_90_per = CreateNewTS(TS_to_Gen, Remove_Sets)
np.save(save_Dataset_Folder + "TS_to_Gen_alpha_90_per.npy", TS_to_Gen_alpha_90_per)

Removed Sequences
['ori' 'mov' 'andi']
['ldsPT2' 'ori' 'mov' 'andi']
['ldsPT1' 'ldsPT2' 'ori' 'mov' 'andi']


In [121]:
sequence = 0
for info in TS_to_Gen_alpha_90_per:
    print("Sequence: ", sequence)
    print(info["Code"])
    print(info["Seq"].shape)
    print("")
    sequence = sequence + 1
    
print("==============================================")
print("Total differing sequences: ", len(TS_to_Gen_alpha_90_per))

Sequence:  0
['nop' 'nop' 'ori']
(6000, 1)

Sequence:  1
['nop' 'ori' 'stsPT1']
(6000, 1)

Sequence:  2
['ori' 'stsPT1' 'stsPT2']
(90000, 1)

Sequence:  3
['stsPT1' 'stsPT2' 'ldsPT1']
(45000, 1)

Sequence:  4
['stsPT2' 'ldsPT1' 'ldsPT2']
(45000, 1)

Sequence:  5
['ldsPT1' 'ldsPT2' 'ori']
(108000, 1)

Sequence:  6
['ldsPT2' 'ori' 'mov']
(21000, 1)

Sequence:  7
['mov' 'andi' 'ori']
(9000, 1)

Sequence:  8
['andi' 'ori' 'stsPT1']
(6000, 1)

Sequence:  9
['stsPT1' 'stsPT2' 'out']
(9000, 1)

Sequence:  10
['stsPT2' 'out' 'ldsPT1']
(9000, 1)

Sequence:  11
['out' 'ldsPT1' 'ldsPT2']
(33000, 1)

Sequence:  12
['ldsPT2' 'ori' 'stsPT1']
(60000, 1)

Sequence:  13
['mov' 'andi' 'pushPT1']
(6000, 1)

Sequence:  14
['andi' 'pushPT1' 'pushPT2']
(6000, 1)

Sequence:  15
['pushPT1' 'pushPT2' 'ori']
(27000, 1)

Sequence:  16
['pushPT2' 'ori' 'mov']
(18000, 1)

Sequence:  17
['mov' 'andi' 'or']
(9000, 1)

Sequence:  18
['andi' 'or' 'pushPT1']
(3000, 1)

Sequence:  19
['or' 'pushPT1' 'pushPT2']
(3000, 1)